# M0 · 05 — Matrix multiply (`@`)

`@` is matrix multiplication — the single most important op in the model.
`nn.Linear` is a matmul; attention scores `q @ kᵀ` are a matmul; combining
values `weights @ v` is a matmul.

**The rule:** `(n, k) @ (k, m) → (n, m)`. The inner dimensions must match and
cancel; the outer ones survive.

In [ ]:
import torch
from m0_checks import check, check_tensor, TODO
torch.manual_seed(0)

## 1. What one output element is

`(A @ B)[i, j]` = dot product of **row i of A** with **column j of B**.
Compute `A @ B` for the small matrices below by calling `@` (then eyeball that
the top-left entry is `1*5 + 2*7 = 19`).

In [ ]:
A = torch.tensor([[1, 2], [3, 4]])
B = torch.tensor([[5, 6], [7, 8]])
C = TODO
C

In [ ]:
check('A @ B', C, torch.tensor([[19, 22], [43, 50]]))

## 2. Inner dims must match

`(2, 3) @ (3, 4) → (2, 4)`: the 3's cancel. Multiply `P (2×3)` by `Q (3×4)`
and check the output shape is `(2, 4)`.

In [ ]:
P = torch.randn(2, 3)
Q = torch.randn(3, 4)
R = TODO
R.shape

In [ ]:
check_tensor('shape (2,4)', R, shape=(2, 4))

## 3. This is what `nn.Linear` does

`nn.Linear(in, out)` holds a weight of shape `(out, in)` and computes
`x @ W.T`. So input `(T, in)` → output `(T, out)`. In the GPT, `self.key(x)`
turns `(B,T,C)` into `(B,T,head_size)` exactly this way.

Given `x (4×3)` and weight `W (5×3)` (out=5, in=3), compute `x @ W.T` → `(4, 5)`.

In [ ]:
x = torch.randn(4, 3)
W = torch.randn(5, 3)  # (out=5, in=3), like nn.Linear(3, 5).weight
y = TODO
y.shape

In [ ]:
check_tensor('linear output (4,5)', y, shape=(4, 5))

## 4. Batched matmul — the extra leading axes ride along

When tensors have >2 axes, `@` multiplies the **last two** and broadcasts the
rest. `(B, T, hs) @ (B, hs, T) → (B, T, T)` — the GPT's attention scores.

Compute `q @ k.transpose(-2, -1)` for `q, k` of shape `(2, 3, 4)`; expect
`(2, 3, 3)`.

In [ ]:
q = torch.randn(2, 3, 4)  # (B, T, hs)
k = torch.randn(2, 3, 4)
scores = TODO
scores.shape

In [ ]:
check_tensor('scores (B,T,T) = (2,3,3)', scores, shape=(2, 3, 3))

## 5. The other attention matmul — `weights @ v`

After softmax, `weights (B,T,T)` combines the value vectors `v (B,T,hs)` into the
output `(B,T,hs)`. Compute it and check the shape.

In [ ]:
weights = torch.softmax(torch.randn(2, 3, 3), dim=-1)  # rows sum to 1
v = torch.randn(2, 3, 4)                               # (B, T, hs)
out = TODO
out.shape

In [ ]:
check_tensor('attention out (2,3,4)', out, shape=(2, 3, 4))

## ✅ Recap

- `@` = matmul; `(n,k)@(k,m)→(n,m)`; inner dims cancel.
- `nn.Linear` = `x @ W.T`; turns `C`→`head_size`.
- Batched: leading axes ride along; `q @ kᵀ → (B,T,T)`, `weights @ v → (B,T,hs)`.
- You have now built **every matmul in the attention head** by hand.

Next: **06 — reductions & softmax** (turning scores into attention weights).